In [9]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
import pandas as pd 
import numpy as np 
import plotly.express  as px 
import plotly.graph_objects as go 

In [2]:
data = load_diabetes(as_frame=True)
df = data.frame

In [3]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
test_df, valid_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [4]:
x_train, y_train = train_df.drop("target", axis=1), train_df["target"]

In [5]:
x_val, y_val = valid_df.drop("target", axis = 1), valid_df["target"]

In [6]:
depth = [4,6,8,10,12,None]
result = []


for d in depth:
    rf = RandomForestRegressor( max_depth=d, random_state=42 )
    rf.fit(x_train,y_train)

    val_pred = rf.predict(x_val)
    train_pred = rf.predict(x_train)


    result.append({
        "max_depth" : d,
        "Train_r2score" : r2_score(y_train,train_pred),
        "Vaildation_r2score" : r2_score(y_val,val_pred),
        "valid_MSE" : mean_squared_error(y_val,val_pred)**0.5,
        "vaild_MAE" : mean_absolute_error(y_val,val_pred)
    })

pd.DataFrame(result)

,max_depth,Train_r2score,Vaildation_r2score,valid_MSE,vaild_MAE
0,4.0,0.666295,0.482487,54.207282,45.977403
1,6.0,0.806407,0.486994,53.970694,45.566703
2,8.0,0.881254,0.493293,53.638336,45.188312
3,10.0,0.907193,0.485058,54.072433,45.546416
4,12.0,0.916427,0.474672,54.615017,46.155258
5,NaN,0.918900,0.484917,54.079865,45.633582


In [7]:
print(pd.DataFrame(result))

   max_depth  Train_r2score  Vaildation_r2score  valid_MSE  vaild_MAE
0        4.0       0.666295            0.482487  54.207282  45.977403
1        6.0       0.806407            0.486994  53.970694  45.566703
2        8.0       0.881254            0.493293  53.638336  45.188312
3       10.0       0.907193            0.485058  54.072433  45.546416
4       12.0       0.916427            0.474672  54.615017  46.155258
5        NaN       0.918900            0.484917  54.079865  45.633582


In [8]:
feature_options = [ "sqrt", "log2", 0.3, 0.5, 1.0]
result = []


for f in feature_options:
    rf = RandomForestRegressor( max_depth=8, random_state=42, max_features=f )
    rf.fit(x_train,y_train)

    val_pred = rf.predict(x_val)
    train_pred = rf.predict(x_train)


    result.append({
        "feature_options" : f,
        "Train_r2score" : r2_score(y_train,train_pred),
        "Vaildation_r2score" : r2_score(y_val,val_pred),
        "valid_MSE" : mean_squared_error(y_val,val_pred)**0.5,
        "vaild_MAE" : mean_absolute_error(y_val,val_pred)
    })

print(pd.DataFrame(result))

  feature_options  Train_r2score  Vaildation_r2score  valid_MSE  vaild_MAE
0            sqrt       0.863899            0.490331  53.794889  45.154621
1            log2       0.863899            0.490331  53.794889  45.154621
2             0.3       0.863899            0.490331  53.794889  45.154621
3             0.5       0.873348            0.479323  54.372720  45.273598
4             1.0       0.881254            0.493293  53.638336  45.188312


In [11]:
param_dist = {
    "n_estimators" : [100,200,300],
    "max_depth" : [6,8,10],
    "max_features" : [1.0,"sqrt","log2"],
    "min_samples_split" : [2,4,6],
    "min_samples_leaf" : [1,2,3]
}


rf = RandomForestRegressor(random_state=42)
random_search = RandomizedSearchCV(estimator=rf,param_distributions=param_dist,
                                   n_iter=20,
                                   scoring="r2",
                                   cv=3,
                                   random_state=42,
                                   n_jobs= -1)

random_search.fit(x_train,y_train)

best_model = random_search.best_estimator_
best_params = random_search.best_params_

val_pred = best_model.predict(x_val)

result = {
    "Best_Params": best_params,
    "Val_R2": r2_score(y_val, val_pred),
    "Val_MAE": mean_absolute_error(y_val, val_pred),
    "Val_RMSE": mean_squared_error(y_val, val_pred) ** 0.5
}

result

{'Best_Params': {'n_estimators': 100,
  'min_samples_split': 6,
  'min_samples_leaf': 2,
  'max_features': 'log2',
  'max_depth': 8},
 'Val_R2': 0.4840450221855034,
 'Val_MAE': 45.15079080530123,
 'Val_RMSE': 54.12560697136944}